# IMPORTING LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger, ModelCheckpoint, ReduceLROnPlateau
import gc

# 1. EXPERIMENT CONFIGURATION

In [2]:
EXPERIMENT_NAME = "LSTM_Exp04"

TRAIN_PATH = "train.parquet"
TEST_PATH = "test.parquet"
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"

TARGET_COL = "PRESSURE"

TIME_STEPS = 60
BATCH_SIZE = 1024
EPOCHS = 30

LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 64
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001

MA_WINDOW = 10
MISSING_DROP_THRESHOLD = 0.95

VALIDATION_RATIO = 0.2

print(f"--- Starting Experiment: {EXPERIMENT_NAME} ---")

--- Starting Experiment: LSTM_Exp04 ---


# 2. CYCLICAL TIME FEATURE FUNCTION

In [3]:
def extract_cyclic_time(df, time_col="LMST"):
    """
    Extracts HH:MM:SS from the LMST column and converts it into sine/cosine features.
    This helps the model understand daily cyclic pressure patterns.
    """

    if time_col not in df.columns:
        print(f" -> {time_col} not found. Skipping cyclic time extraction.")
        return df

    print(f" -> Extracting cyclic time features from {time_col}...")

    time_strings = df[time_col].astype(str).str.extract(r"(\d{2}:\d{2}:\d{2})")[0]

    parsed_times = pd.to_datetime(
        time_strings,
        format="%H:%M:%S",
        errors="coerce"
    )

    hour_of_sol = (
        parsed_times.dt.hour
        + parsed_times.dt.minute / 60.0
        + parsed_times.dt.second / 3600.0
    )

    df["Time_Sin"] = np.sin(2 * np.pi * hour_of_sol / 24.0)
    df["Time_Cos"] = np.cos(2 * np.pi * hour_of_sol / 24.0)

    return df

# 3. FEATURE CLEANING FUNCTION

In [4]:
def preprocess_train_test(train_df, test_df):
    """
    Applies the same preprocessing to train and test data:
    - cyclic time features
    - missing indicators
    - bad column removal
    - interpolation
    - selective moving average smoothing
    """

    print("Applying cyclical time encoding...")
    train_df = extract_cyclic_time(train_df, "LMST")
    test_df = extract_cyclic_time(test_df, "LMST")

    print("Creating missing-value indicators...")

    original_feature_cols = [
        col for col in train_df.columns
        if col != TARGET_COL
    ]

    cols_with_missing = [
        col for col in original_feature_cols
        if train_df[col].isnull().any() or test_df[col].isnull().any()
    ]

    for col in cols_with_missing:
        train_df[f"{col}_missing"] = train_df[col].isnull().astype(np.int8)
        test_df[f"{col}_missing"] = test_df[col].isnull().astype(np.int8)

    print("Finding columns with too much missing data...")

    missing_percentages = train_df.isnull().mean()

    high_missing_cols = missing_percentages[
        missing_percentages > MISSING_DROP_THRESHOLD
    ].index.tolist()

    manual_drop_cols = [
        "LMST",
        "LTST",
        "SCLK"
    ]

    columns_to_drop = list(set(high_missing_cols + manual_drop_cols))

    if TARGET_COL in columns_to_drop:
        columns_to_drop.remove(TARGET_COL)

    print("Dropping columns:")
    print(columns_to_drop)

    train_df = train_df.drop(columns=columns_to_drop, errors="ignore")
    test_df = test_df.drop(columns=columns_to_drop, errors="ignore")

    print("Converting non-numeric columns if needed...")

    for col in train_df.columns:
        if col == TARGET_COL:
            continue

        if train_df[col].dtype == "object":
            train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
            test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

    print("Interpolating missing values...")

    train_df = train_df.interpolate(method="linear").ffill().bfill()
    test_df = test_df.interpolate(method="linear").ffill().bfill()

    print("Applying selective moving average smoothing...")

    feature_cols = [
        col for col in train_df.columns
        if col != TARGET_COL
    ]

    do_not_smooth_keywords = [
        "missing",
        "FOV",
        "OFF",
        "STILL",
        "TILT",
        "TRANSDUCER",
        "Time_Sin",
        "Time_Cos",
        "sol"
    ]

    smooth_cols = []

    for col in feature_cols:
        should_not_smooth = any(keyword in col for keyword in do_not_smooth_keywords)

        if not should_not_smooth:
            smooth_cols.append(col)

    print(f"Number of smoothed columns: {len(smooth_cols)}")
    print(f"Number of unsmoothed columns: {len(feature_cols) - len(smooth_cols)}")

    train_df[smooth_cols] = train_df[smooth_cols].rolling(
        window=MA_WINDOW,
        min_periods=1
    ).mean()

    test_df[smooth_cols] = test_df[smooth_cols].rolling(
        window=MA_WINDOW,
        min_periods=1
    ).mean()

    final_features = [
        col for col in train_df.columns
        if col != TARGET_COL
    ]

    return train_df, test_df, final_features

# 4. LOAD DATA

In [5]:
print("Loading datasets...")

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Loading datasets...
Train shape: (8130533, 36)
Test shape: (3949990, 35)


# 5. PREPROCESS DATA

In [6]:
train_df, test_df, final_features = preprocess_train_test(train_df, test_df)

print("Final number of features:", len(final_features))
print("Final features:")
print(final_features)

Applying cyclical time encoding...
 -> Extracting cyclic time features from LMST...
 -> Extracting cyclic time features from LMST...
Creating missing-value indicators...
Finding columns with too much missing data...
Dropping columns:
['SCLK', 'VOLUME_MIXING_RATIO_UNCERTAINTY', 'SKYCAM_OFF', 'WHEEL_OUTSIDE_TIRS_DOWNWARD_LOOKING_FOV', 'LTST', 'LOCAL_RELATIVE_HUMIDITY_UNCERTAINTY', 'MASTHEAD_AZIMUTH', 'ROVER_HGA_OFF', 'PRESSURE_UNCERTAINTY', 'LMST', 'MASTHEAD_ELEVATION']
Converting non-numeric columns if needed...
Interpolating missing values...
Applying selective moving average smoothing...
Number of smoothed columns: 18
Number of unsmoothed columns: 29
Final number of features: 47
Final features:
['SOLAR_LONGITUDE_ANGLE', 'SOLAR_ZENITHAL_ANGLE', 'ROVER_POSITION_X', 'ROVER_POSITION_Y', 'ROVER_POSITION_Z', 'ROVER_VELOCITY', 'ROVER_PITCH', 'ROVER_YAW', 'ROVER_ROLL', 'sol', 'TRANSDUCER', 'LOCAL_RELATIVE_HUMIDITY', 'HUMIDITY_LOCAL_TEMP', 'HUMIDITY_LOCAL_TEMP_UNCERTAINTY', 'VOLUME_MIXING_RATI

# 6. SPLIT TRAIN / VALIDATION

In [7]:
print("Preparing raw arrays...")

X_all = train_df[final_features].values.astype(np.float32)
y_all = train_df[TARGET_COL].values.astype(np.float32)
X_test_raw = test_df[final_features].values.astype(np.float32)

num_rows = len(X_all)
split_idx = int(num_rows * (1.0 - VALIDATION_RATIO))

print("Total training rows:", num_rows)
print("Train split rows:", split_idx)
print("Validation rows:", num_rows - split_idx)

del train_df
del test_df
gc.collect()

Preparing raw arrays...
Total training rows: 8130533
Train split rows: 6504426
Validation rows: 1626107


0

# 7. SCALE DATA

In [8]:
print("Scaling data...")

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_all[:split_idx])
X_val_scaled = scaler.transform(X_all[split_idx - TIME_STEPS + 1:])
X_test_scaled = scaler.transform(X_test_raw)

y_train = y_all[TIME_STEPS - 1:split_idx]
y_val = y_all[split_idx:]

print("X_train_scaled shape:", X_train_scaled.shape)
print("y_train shape:", y_train.shape)

print("X_val_scaled shape:", X_val_scaled.shape)
print("y_val shape:", y_val.shape)

del X_all
del X_test_raw
gc.collect()

Scaling data...
X_train_scaled shape: (6504426, 47)
y_train shape: (6504367,)
X_val_scaled shape: (1626166, 47)
y_val shape: (1626107,)


0

# 8. CREATE TIME-SERIES DATASETS

In [9]:
print("Creating TensorFlow time-series datasets...")

train_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X_train_scaled,
    targets=y_train,
    sequence_length=TIME_STEPS,
    sequence_stride=1,
    sampling_rate=1,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X_val_scaled,
    targets=y_val,
    sequence_length=TIME_STEPS,
    sequence_stride=1,
    sampling_rate=1,
    batch_size=BATCH_SIZE,
    shuffle=False
)

train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

Creating TensorFlow time-series datasets...


# 9. BUILD MODEL

In [10]:
print("Building LSTM model...")

model = Sequential([
    Input(shape=(TIME_STEPS, len(final_features))),

    LSTM(LSTM_UNITS_1, return_sequences=True),
    Dropout(DROPOUT_RATE),

    LSTM(LSTM_UNITS_2),
    Dropout(DROPOUT_RATE),

    Dense(64, activation="relu"),
    Dense(1)
])

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=["mae"]
)

model.summary()

Building LSTM model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 128)        │        90,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 143,745 (561.50 KB)

 Trainable params: 143,745 (561.50 KB)

 Non-trainable params: 0 (0.00 B)

# 10. CALLBACKS

In [11]:
csv_logger = CSVLogger(
    f"{EXPERIMENT_NAME}_log.csv",
    append=True
)

checkpoint = ModelCheckpoint(
    f"{EXPERIMENT_NAME}_best_model.keras",
    save_best_only=True,
    monitor="val_loss",
    mode="min",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    mode="min",
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    mode="min",
    verbose=1
)

# 11. TRAIN MODEL

In [12]:
print("Starting training...")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[
        early_stop,
        reduce_lr,
        csv_logger,
        checkpoint
    ]
)

Starting training...
Epoch 1/30
6352/6352 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step - loss: 70312.8781 - mae: 147.0637
Epoch 1: val_loss improved from None to 30.38893, saving model to LSTM_Exp04_best_model.keras

Epoch 1: finished saving model to LSTM_Exp04_best_model.keras
6352/6352 ━━━━━━━━━━━━━━━━━━━━ 2527s 397ms/step - loss: 17689.3242 - mae: 63.3774 - val_loss: 30.3889 - val_mae: 4.5205 - learning_rate: 0.0010
Epoch 2/30
6352/6352 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step - loss: 1918.8278 - mae: 34.9140
Epoch 2: val_loss did not improve from 30.38893
6352/6352 ━━━━━━━━━━━━━━━━━━━━ 2516s 396ms/step - loss: 1795.9188 - mae: 33.7623 - val_loss: 125.2918 - val_mae: 10.2204 - learning_rate: 0.0010
Epoch 3/30
6352/6352 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step - loss: 1286.5542 - mae: 28.5012
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 3: val_loss did not improve from 30.38893
6352/6352 ━━━━━━━━━━━━━━━━━━━━ 2537s 399ms/step - loss: 1070.1042 - mae: 25.9076 - val_loss

# 12. RE-SCALE USING FULL TRAINING DATA FOR FINAL SUBMISSION

In [13]:
print("Re-loading full data for final model scaling...")

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

train_df, test_df, final_features = preprocess_train_test(train_df, test_df)

X_full_raw = train_df[final_features].values.astype(np.float32)
X_test_raw = test_df[final_features].values.astype(np.float32)

del train_df
del test_df
gc.collect()

print("Fitting final scaler on full training data...")

final_scaler = MinMaxScaler()
X_full_scaled = final_scaler.fit_transform(X_full_raw)
X_test_scaled_final = final_scaler.transform(X_test_raw)

del X_full_raw
del X_test_raw
gc.collect()

Re-loading full data for final model scaling...
Applying cyclical time encoding...
 -> Extracting cyclic time features from LMST...
 -> Extracting cyclic time features from LMST...
Creating missing-value indicators...
Finding columns with too much missing data...
Dropping columns:
['SCLK', 'VOLUME_MIXING_RATIO_UNCERTAINTY', 'SKYCAM_OFF', 'WHEEL_OUTSIDE_TIRS_DOWNWARD_LOOKING_FOV', 'LTST', 'LOCAL_RELATIVE_HUMIDITY_UNCERTAINTY', 'MASTHEAD_AZIMUTH', 'ROVER_HGA_OFF', 'PRESSURE_UNCERTAINTY', 'LMST', 'MASTHEAD_ELEVATION']
Converting non-numeric columns if needed...
Interpolating missing values...
Applying selective moving average smoothing...
Number of smoothed columns: 18
Number of unsmoothed columns: 29
Fitting final scaler on full training data...


0

# 13. PREPARE TEST SEQUENCES

In [14]:
print("Preparing test sequences...")

last_train_context = X_full_scaled[-TIME_STEPS + 1:]
X_test_padded = np.vstack([last_train_context, X_test_scaled_final])

test_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X_test_padded,
    targets=None,
    sequence_length=TIME_STEPS,
    sequence_stride=1,
    sampling_rate=1,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

Preparing test sequences...


# 14. GENERATE PREDICTIONS

In [15]:
print("Generating predictions...")

predictions = model.predict(test_dataset)

predictions = predictions.flatten()

print("Prediction count:", len(predictions))

Generating predictions...
3858/3858 ━━━━━━━━━━━━━━━━━━━━ 737s 190ms/step
Prediction count: 3949990


# 15. CREATE SUBMISSION

In [16]:
submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Submission row count:", len(submission))

if len(predictions) != len(submission):
    raise ValueError(
        f"Prediction length mismatch. "
        f"Predictions: {len(predictions)}, Submission rows: {len(submission)}"
    )

submission[TARGET_COL] = predictions

submission_filename = f"submission_{EXPERIMENT_NAME}.csv"

submission.to_csv(
    submission_filename,
    index=False
)

print(f"Experiment complete.")
print(f"Saved submission: {submission_filename}")
print(f"Saved training log: {EXPERIMENT_NAME}_log.csv")
print(f"Saved best model: {EXPERIMENT_NAME}_best_model.keras")

Submission row count: 3949990
Experiment complete.
Saved submission: submission_LSTM_Exp04.csv
Saved training log: LSTM_Exp04_log.csv
Saved best model: LSTM_Exp04_best_model.keras
